# 06 — Trích xuất 12 features VCG cho train / valid / test

Notebook này **trực tiếp chạy pipeline V1 trên từng recording** của cả ba tập,
rồi xuất **ba file CSV riêng**:

- `outputs/vcg_12_features_train_valid_test/train_features.csv`
- `outputs/vcg_12_features_train_valid_test/valid_features.csv`
- `outputs/vcg_12_features_train_valid_test/test_features.csv`

Mở notebook tại thư mục project, chọn kernel **ipcv** rồi **Run All**.
Các ô đã được chạy và lưu output. Mặc định dùng 30 giây đầu mỗi recording.
Lần chạy lại dùng checkpoints khi cấu hình, source, dữ liệu và split không đổi.
Muốn thử cấu hình khác, đổi `OUTPUT_DIR` sang thư mục mới.

Mỗi dòng CSV là **một recording**, có patient ID, recording ID, split,
chẩn đoán gốc, nhãn **HC = 0 / MI = 1**, 12 features, biên QRS/T và QC.
Giữ các record `needs_review`; giá trị không tính được giữ **NaN kèm nguyên nhân**.
Trích xuất đủ 12 features hoặc qua QC không đồng nghĩa biên đã được xác nhận đúng.
Notebook chưa huấn luyện mô hình phân loại.

## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(6)

## 1. Import và cấu hình

Thuật toán dùng đúng các hàm V1 mà notebook 05 sử dụng. Hàm `run_staged()`
gọi lần lượt các hàm đó và giữ chẩn đoán của bước đã hoàn tất nếu bước sau lỗi.
Không thay threshold, tiếp tuyến, bộ lọc hoặc quy tắc phân đoạn.

In [ ]:
import importlib.metadata
from pathlib import Path
from dataclasses import asdict
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'vcg_pipeline.py').is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'scripts'))
from vcg_preprocessing import Config
from vcg_pipeline import FEATURE_NAMES
from run_full_vcg_evaluation import (
    discover_records, make_split, run_staged, describe_result,
    write_json, jsonable, sha256,
)

DATA_ROOT = ROOT/'content/ptbdb/physionet.org/files/ptbdb/1.0.0'
OUTPUT_DIR = ROOT/'outputs/vcg_12_features_train_valid_test'
EXISTING_SPLIT = ROOT/'outputs/vcg_full_evaluation/split_manifest.csv'
CFG = Config(seconds=30.0)
RESUME = True

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)
print('Data:', DATA_ROOT)
print('CSV output:', OUTPUT_DIR)
print('Số features:', len(FEATURE_NAMES))
display(pd.DataFrame({'feature': FEATURE_NAMES}))

## 2. Đọc metadata thật và tái sử dụng split theo bệnh nhân

Nhãn bệnh được đọc từ header gốc. Chỉ HC và MI thuộc phạm vi trích xuất này.
Ưu tiên manifest của lần đánh giá toàn bộ PTB đã có. Nếu chưa có manifest,
tái lập split cũ notebook 02 từ bảng features cũ, giữ nguyên test và tách
validation từ train cũ. Tên `val` được đổi thành **`valid`** trong notebook này.

Các record từng lỗi vẫn thuộc tập của bệnh nhân đó. Split được tạo trước khi
trích xuất, không phụ thuộc QC hay số features tính được.

In [ ]:
inventory = discover_records(DATA_ROOT)
manifest, split_details = make_split(
    inventory,
    existing_features=ROOT/'vcg_12_features.csv',
    existing_manifest=EXISTING_SPLIT if EXISTING_SPLIT.is_file() else None,
)
manifest['split'] = manifest['split'].replace({'val': 'valid', 'validation': 'valid'})
manifest = manifest.sort_values('record_id').reset_index(drop=True)

assert manifest.record_id.is_unique
assert manifest.groupby('patient_id').split.nunique().eq(1).all()
assert set(manifest.split) == {'train', 'valid', 'test'}
assert set(manifest.record_id) == set(inventory.loc[inventory.included, 'record_id'])
patient_sets = {name: set(group.patient_id) for name, group in manifest.groupby('split')}
assert not (patient_sets['train'] & patient_sets['valid'])
assert not (patient_sets['train'] & patient_sets['test'])
assert not (patient_sets['valid'] & patient_sets['test'])

print(f'Toàn dataset: {len(inventory)} recordings / {inventory.patient_id.nunique()} bệnh nhân')
print(f'HC/MI cần chạy: {len(manifest)} recordings / {manifest.patient_id.nunique()} bệnh nhân')
display(manifest.groupby(['split', 'label_name']).agg(
    patients=('patient_id', 'nunique'), recordings=('record_id', 'size')))
display(manifest.groupby('split').agg(
    patients=('patient_id', 'nunique'), recordings=('record_id', 'size')).reindex(['train', 'valid', 'test']))

## 3. Khóa cấu hình, lưu manifest và chuẩn bị checkpoint

Notebook kiểm tra hash module V1, hàm batch, dữ liệu header/tín hiệu, split và
phiên bản thư viện trước khi resume. Không ghi đè kết quả của cấu hình khác.
Ba CSV được ghi từ dữ liệu vừa trích xuất hoặc checkpoints cùng lần chạy.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR/'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

def save_csv_safely(frame, path):
    content = frame.to_csv(index=False)
    if path.exists():
        if path.read_text(encoding='utf-8') != content:
            raise FileExistsError(f'{path} đã có nội dung khác. Đổi OUTPUT_DIR cho thí nghiệm mới.')
        return
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(content, encoding='utf-8')
    temporary.replace(path)

source_names = ['vcg_pipeline.py', 'vcg_preprocessing.py', 'scripts/run_full_vcg_evaluation.py']
run_config = jsonable({
    'baseline': 'V1 unchanged',
    'config': asdict(CFG),
    'source_sha256': {name: sha256(ROOT/name) for name in source_names},
    'inventory_sha256': hashlib.sha256(inventory.to_csv(index=False).encode()).hexdigest(),
    'split_sha256': hashlib.sha256(manifest.to_csv(index=False).encode()).hexdigest(),
    'versions': {name: importlib.metadata.version(name)
                 for name in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'wfdb']},
    'label_mapping': {'HC': 0, 'MI': 1},
    'boundary_time_reference': 'median_relative_ms',
    'missing_features': 'NaN; no imputation',
})
config_path = OUTPUT_DIR/'config.json'
if config_path.exists():
    if json.loads(config_path.read_text()) != run_config:
        raise ValueError('Source/config/data/split/library version đã thay đổi. Chọn OUTPUT_DIR mới.')
    if not RESUME:
        raise FileExistsError('Thư mục đã có lần chạy. Bật RESUME hoặc chọn OUTPUT_DIR mới.')
else:
    if list(CHECKPOINT_DIR.glob('*.json')):
        raise ValueError('Có checkpoints nhưng không có config để xác minh.')
    write_json(config_path, run_config)

save_csv_safely(manifest, OUTPUT_DIR/'split_manifest.csv')
print('Đã lưu cấu hình và manifest; các bệnh nhân không trùng giữa ba tập.')

## 4. Hàm trích xuất cho một tập

Mỗi recording chạy:

`Frank XYZ → input QC → preprocessing → QRS anchors → median beat → QRS/T
delineation → cắt loops → 12 features → QC → checkpoint`.

Đạo hàm vận tốc được tính trên toàn median beat rồi mới cắt theo loop, giữ
đúng V1. Không bỏ record chỉ vì `needs_review`. Khi một record lỗi, notebook
lưu NaN cùng bước lỗi, rồi tiếp tục record kế tiếp.

Các biên thời gian trong CSV tính bằng **ms so với anchor của median beat**;
QRS-on âm là bình thường trong hệ quy chiếu này.

In [ ]:
LEADING_COLUMNS = ['patient_id', 'record_id', 'split', 'diagnosis', 'label', 'label_name'] + FEATURE_NAMES

def extract_split(split_name):
    selected = manifest.loc[manifest['split'].eq(split_name)]
    rows = []
    resumed = 0
    for metadata in tqdm(selected.to_dict('records'), desc=f'Extract {split_name}', unit='record', mininterval=1.0):
        rid = metadata['record_id']
        checkpoint = CHECKPOINT_DIR/(rid.replace('/', '_')+'.json')
        if RESUME and checkpoint.is_file():
            row = json.loads(checkpoint.read_text())
            assert row['record_id'] == rid and row['split'] == split_name
            resumed += 1
        else:
            trace = {}
            try:
                result = run_staged(DATA_ROOT/rid, CFG, trace)
                row = describe_result(metadata, trace, result)
            except Exception as exc:
                row = describe_result(metadata, trace, error=f'{type(exc).__name__}: {exc}')
            row['boundary_time_reference'] = 'median_relative_ms'
            write_json(checkpoint, row)
        rows.append(row)

    frame = pd.DataFrame(rows)
    frame['label'] = frame['label'].astype(int)
    frame[FEATURE_NAMES] = frame[FEATURE_NAMES].apply(pd.to_numeric, errors='coerce')
    frame = frame.reindex(columns=LEADING_COLUMNS + [c for c in frame if c not in LEADING_COLUMNS])
    frame = frame.sort_values('record_id').reset_index(drop=True)
    assert frame.record_id.is_unique and len(frame) == len(selected)
    assert frame.groupby('patient_id').split.nunique().eq(1).all()
    assert np.isfinite(frame[FEATURE_NAMES]).sum(axis=1).eq(frame.feature_completeness).all()
    assert frame.extraction_success.eq(frame.feature_completeness.eq(len(FEATURE_NAMES))).all()

    target = OUTPUT_DIR/f'{split_name}_features.csv'
    save_csv_safely(frame, target)
    n_full = int(frame.extraction_success.sum())
    print(f'{split_name}: {len(frame)} recordings; {n_full} đủ 12 features; '
          f'{len(frame)-n_full} thiếu features; {resumed} dùng checkpoint.')
    print('Đã xuất:', target)
    display(frame[['record_id', 'label_name', 'feature_completeness', 'qc_status', 'failure_stage']].head())
    return frame

## 5. Trích xuất tập TRAIN → `train_features.csv`

In [ ]:
train_df = extract_split('train')

## 6. Trích xuất tập VALID → `valid_features.csv`

In [ ]:
valid_df = extract_split('valid')

## 7. Trích xuất tập TEST → `test_features.csv`

Cùng cấu hình đã cố định cho cả ba tập. Không dùng test để điều chỉnh ngưỡng
hoặc quy tắc delineation.

In [ ]:
test_df = extract_split('test')

## 8. Kiểm tra ba CSV và thống kê completeness / QC

Các dòng có NaN vẫn được giữ trong CSV tương ứng; chưa impute hoặc loại record.
`extraction_success=True` nghĩa là đủ 12 giá trị hữu hạn.
`qc_status=auto_candidate` nghĩa là không có cờ QC của V1, chưa xác nhận accuracy.

In [ ]:
all_extracted = pd.concat([train_df, valid_df, test_df], ignore_index=True)
assert len(all_extracted) == len(manifest)
assert all_extracted.record_id.is_unique
assert set(all_extracted.record_id) == set(manifest.record_id)
assert all_extracted.groupby('patient_id').split.nunique().eq(1).all()
assert set(all_extracted.label) == {0, 1}

summary = all_extracted.groupby(['split', 'label_name']).agg(
    recordings=('record_id', 'size'),
    patients=('patient_id', 'nunique'),
    full_12_features=('extraction_success', 'sum'),
    needs_review=('qc_status', lambda values: values.eq('needs_review').sum()),
    processing_failed=('processing_success', lambda values: (~values).sum()),
)
summary['full_12_pct'] = 100*summary.full_12_features/summary.recordings
display(summary)

for split_name, frame in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
    saved = pd.read_csv(OUTPUT_DIR/f'{split_name}_features.csv')
    assert len(saved) == len(frame)
    assert all(name in saved for name in FEATURE_NAMES)
    assert saved['split'].eq(split_name).all()
    assert set(saved.record_id) == set(frame.record_id)
    np.testing.assert_allclose(saved[FEATURE_NAMES].to_numpy(), frame[FEATURE_NAMES].to_numpy(),
                               rtol=1e-10, atol=1e-12, equal_nan=True)

display(all_extracted.loc[all_extracted.feature_completeness.lt(len(FEATURE_NAMES)), [
    'record_id', 'split', 'label_name', 'feature_completeness',
    'failure_stage', 'error', 'feature_missing_reasons', 'qc_flags',
]])
print(f'Đã kiểm tra đủ {len(all_extracted)} rows, không rò rỉ bệnh nhân, ba CSV có đủ 12 cột features.')

## 9. Đối chiếu với lần đánh giá toàn bộ V1 đã lưu

Đây là kiểm tra tính nhất quán của triển khai, **không phải đánh giá accuracy**
so với annotation chuyên gia. Notebook vẫn tự trích xuất ở các ô train/valid/test;
bảng đánh giá cũ chỉ được đọc ở bước này để đối chiếu.

In [ ]:
previous_path = ROOT/'outputs/vcg_full_evaluation/features/all_features.csv'
if previous_path.is_file():
    previous = pd.read_csv(previous_path).set_index('record_id')
    current = all_extracted.set_index('record_id').sort_index()
    previous = previous.loc[current.index]
    np.testing.assert_allclose(current[FEATURE_NAMES].to_numpy(), previous[FEATURE_NAMES].to_numpy(),
                               rtol=1e-10, atol=1e-12, equal_nan=True)
    assert current.qc_status.eq(previous.qc_status).all()
    assert current.feature_completeness.eq(previous.feature_completeness).all()
    assert current['split'].eq(previous['split'].replace({'val': 'valid'})).all()
    print(f'{len(current)}/{len(current)} records khớp 12 features, QC, completeness và split với V1 đã đánh giá.')
else:
    print('Chưa có bảng đánh giá cũ để đối chiếu; đã kiểm tra tính nhất quán ba CSV ở bước 8.')

## 10. Ba file đầu ra

Các DataFrame `train_df`, `valid_df`, `test_df` chứa đầy đủ dữ liệu để kiểm tra
tiếp trong notebook. Dùng `FEATURE_NAMES` để chọn đúng 12 cột làm đầu vào ML;
không đưa metadata/nhãn/QC/biên thời gian vào ma trận features một cách mặc định.
Chính sách xử lý missing/QC và các phép fit preprocessing ML cần được xác định
trên train/valid trước khi đánh giá test.

In [ ]:
output_rows = []
for split_name in ['train', 'valid', 'test']:
    path = OUTPUT_DIR/f'{split_name}_features.csv'
    output_rows.append({'split': split_name, 'file': path.name,
                        'rows': len(pd.read_csv(path)), 'bytes': path.stat().st_size,
                        'sha256': sha256(path)})
display(pd.DataFrame(output_rows))
for item in output_rows:
    display(Markdown(f"- [{item['file']}](outputs/vcg_12_features_train_valid_test/{item['file']}) — {item['rows']} recordings"))
write_json(OUTPUT_DIR/'completion.json', {
    'completed_records': len(all_extracted),
    'expected_records': len(manifest),
    'full_12_features': int(all_extracted.extraction_success.sum()),
    'patient_leakage': False,
    'files': output_rows,
})